# 额外的周末练习 —— 第 2 周

## 练习目标（理念）

用第 2 周学到的能力，把第 1 周的「技术问答器」做成完整原型：

- **Gradio UI**：浏览器里聊天，而不是只在终端 `print`
- **流式 / 多模态体验**：文字回答 +（可选）语音播报、配图
- **System Prompt**：用系统提示注入「技术专家 / 老师」人设与回答规范
- **模型切换能力**：作业要求里提到可在模型间切换（本实现集中用一个 `MODEL` 常量）
- **奖励分**：演示 **Tool Use（工具调用）**——让模型决定何时生成配图
- **大胆加分**：音频输入（语音转文字）+ 音频输出（文字转语音）

## 和本课第 2 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Gradio `Blocks` / 事件链 | `text_input.submit(...).then(chat, ...)` |
| System Prompt | `system_message` 定专家角色与配图讲解方式 |
| Function Calling / Tools | `tools` + `handle_tool_calls` + `finish_reason=="tool_calls"` |
| 图像生成 | `openai.images.generate`（DALL·E） |
| 语音 I/O | Whisper 转写 + TTS 朗读 |

## 怎么跑

1. 准备好 `.env`（`OPENAI_API_KEY` 等），从上到下运行单元格
2. 浏览器打开 Gradio；可用文字框提问，或用麦克风说话（会转写进文字框）
3. 模型若调用配图工具，右侧会出图；同时下方会播报语音回答


In [66]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：调用 Chat / Images / Audio 等 API
from openai import OpenAI
# 导入 gradio：快速搭浏览器 UI（Chatbot、Audio、Image 等组件）
import gradio as gr
# 导入标准库 base64：DALL·E 若返回 b64_json，需要解码成二进制图片
import base64
# 从 io 导入 BytesIO：把内存里的字节当成「文件」交给 PIL 打开
from io import BytesIO
# 从 PIL 导入 Image：把解码后的字节变成可在 Gradio 里显示的图片对象
from PIL import Image
# 导入标准库 json：解析 tool call 里的 arguments（JSON 字符串 → Python 字典）
import json


In [67]:
# ========== 初始化：读密钥、建客户端、选定模型 ==========

# 加载 .env；override=True 表示用文件里的值覆盖已有环境变量
load_dotenv(override=True)
# 创建默认 OpenAI 客户端（会从环境变量读 OPENAI_API_KEY）
openai = OpenAI()
# 本练习选用的聊天模型 id（字符串保持原样，改译会指错模型）
MODEL = 'gpt-5'


In [68]:
# ========== System Prompt：定人设 + 要求用配图工具 ==========

# system_message 会作为 role=system 放进 messages 最前面
# 提示词正文保持英文：这是发给模型的指令，翻译会改变行为/风格
system_message = """
        You are a helpful and friendly technical expert and teacher. When a student asks a question, answer it clearly and in detail, using practical examples.
        You should use the image generation tool to create an illustration that helps to visualize the concept.
        First, answer the student's question directly. Then, explain the generated image in the context of your explanation. Tell the student how the image relates to the concept, rather than giving a dry, technical description of the image's appearance or the prompt used to create it. Reply in Markdown, make your respond consistent like a statement.
                """


In [69]:
# ========== 工具实现：根据英文 image_prompt 调用 DALL·E 生成图 ==========

def generate_image(image_prompt):
    # 调用 Images API：model / prompt / size / n / response_format 参数保持原样（影响行为）
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=image_prompt,
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    # 取出第一张图的 base64 字符串（不含 data: 前缀）
    image_base64 = image_response.data[0].b64_json
    # base64 → 原始 PNG/JPEG 字节
    image_data = base64.b64decode(image_base64)
    # BytesIO 包一层，让 PIL 像读文件一样打开，返回 Image 对象给 Gradio
    return Image.open(BytesIO(image_data))


In [70]:
# ========== TTS：把助手文字回答合成语音字节 ==========

def generate_voice_respond(message):
    # 调用音频语音合成 API；model / voice 字符串保持原样
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",
      input=message
    )
    # .content 是音频二进制，可直接交给 Gradio Audio 组件播放
    return response.content


In [71]:
# ========== STT：把麦克风录下的音频文件转成文字 ==========

def transcribe(audio_path):
    # Gradio 未录音时 filepath 可能是 None，直接返回空串，避免打开失败
    if audio_path is None:
        return ""
    # 以二进制只读打开临时音频文件，交给 Whisper 转写
    with open(audio_path, "rb") as audio_file:
        transcript = openai.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file
        )
    # 返回转写文本，后面会写进文字输入框
    return transcript.text


In [72]:
# ========== Tool Schema：告诉模型「有一个 generate_image 可调」 ==========

# OpenAI function calling 需要 JSON Schema 描述函数名、参数、是否必填
image_function = {
    "name": "generate_image",
    # description / parameters 里的英文是给模型看的工具说明，保留原文以免改工具语义
    "description": "Enter a prompt to generate image.",
    "parameters": {
        "type": "object",
        "properties": {
            "image_prompt": {
                "type": "string",
                "description": "The explanation of image",
            },
        },
        "required": ["image_prompt"],
        "additionalProperties": False
    }
}
# Chat Completions 的 tools 参数格式：type=function + function 定义
tools = [{"type": "function", "function": image_function}]


In [73]:
# ========== 处理 tool_calls：真正执行配图并组装 tool 回执 ==========

def handle_tool_calls(message):
    # 本实现只取第一条 tool call（作业演示够用；多工具时可循环）
    tool_call = message.tool_calls[0]
    # 按函数名分发；这里只实现 generate_image
    if tool_call.function.name == 'generate_image':
        # arguments 是 JSON 字符串，解析出 image_prompt
        arguments = json.loads(tool_call.function.arguments)
        image_prompt = arguments.get('image_prompt')
        # 真正调用上面的 generate_image，得到 PIL Image
        image = generate_image(image_prompt)
        # 组装 role=tool 的消息；额外塞 image 字段给上层 UI（稍后会 pop 掉再发给 API）
        response = {
            "role": "tool",
            # content 给模型看的回执文案，保留英文原样
            "content": "Image generated successfully.",
            "tool_call_id": tool_call.id,
            "image": image
        }
        return response


In [74]:
# ========== 核心对话：带 tools 的 Chat Completions 循环 ==========

def chat(history):
    # Gradio messages 格式可能带多余字段；只保留 role + content 再交给 API
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # system 在最前，后面接完整聊天历史
    messages = [{"role": "system", "content": system_message}] + history
    # 第一次请求：带上 tools，让模型可以发起 function call
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools = tools)
    # image 默认没有；若走了配图工具再赋值
    image = None

    # 若 finish_reason 是 tool_calls，就进入「执行工具 → 把结果塞回 messages → 再问模型」循环
    while response.choices[0].finish_reason=="tool_calls":
         message = response.choices[0].message
         tool_response = handle_tool_calls(message)
         # 从 tool 回执里取出图片给 Gradio；pop 掉非 API 字段，避免把 Image 对象再塞进 messages
         image = tool_response.pop("image")
         # 先追加助手那条带 tool_calls 的 message，再追加 tool 结果
         messages.append(message)
         messages.append(tool_response)
         # 再次调用：模型看到「图已生成」后写出最终文字回答
         response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # 取出最终助手文本
    reply = response.choices[0].message.content
    # 把助手回复追加进 history，供 Chatbot 刷新显示
    history += [{"role":"assistant", "content":reply}]
    # 把同一段文字合成语音，实现「会说话的助教」
    voice_respond = generate_voice_respond(reply)

    # 返回三元组：更新后的对话、音频字节、配图（可能为 None）
    return history, voice_respond, image


In [75]:
# ========== Gradio UI：聊天 + 配图 + 语音输入/输出 ==========

# 提交文字时：清空输入框，并把 user 消息追加进 chatbot history
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# 用 Blocks 自由排版多组件（比 ChatInterface 更灵活）
with gr.Blocks() as ui:
    with gr.Row():
        # 左侧对话区；type='messages' 使用 role/content 消息列表格式
        chatbot = gr.Chatbot(height = 500, type='messages')
        # 右侧展示工具生成的配图；interactive=False 表示用户不能手动画图
        image_output = gr.Image(height = 500, interactive=False)
    with gr.Row():
        # 自动播放 TTS 音频
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        # 文字输入；placeholder / label 是 UI 文案（不影响模型逻辑，这里保留原英文）
        text_input = gr.Textbox(placeholder='Enter your message ...' , label='Chat with our technical AI Assistant:')
    with gr.Row():
        # 麦克风：type="filepath" 表示回调收到的是临时文件路径
        audio_input =  gr.Audio(label='Speak to your ai assistant.', type="filepath")
# 音频变化 → Whisper 转写 → 填进文字框（用户仍可改字再提交）
    audio_input.change(transcribe, inputs=[audio_input], outputs=[text_input])
    # 回车提交：先把 user 消息放进 chatbot，再 .then 调用 chat 生成回复/语音/图
    text_input.submit(put_message_in_chatbot, inputs=[text_input, chatbot], outputs=[text_input, chatbot]).then(
            chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

# 启动 Gradio；inbrowser=True 尝试自动开浏览器；auth 为简单账号密码保护
ui.launch(inbrowser=True, auth=("eryk", "banana"))


* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
